In [0]:
%sql

CREATE OR REPLACE TEMPORARY VIEW encounters_with_previous AS
SELECT 
  encounter_id,
  patient_id,
  start as start_date,
  stop as end_date,
  DATE_TRUNC('month', start) AS encounter_month,
  LAG(stop) OVER (PARTITION BY patient_id ORDER BY start, encounter_id) AS previous_end_date,
  LAG(encounter_id) OVER (PARTITION BY patient_id ORDER BY start, encounter_id) AS previous_encounter_id
FROM medical_pipeline.gold.fact_encounters;




In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW classified_encounters AS
SELECT 
  encounter_id,
  patient_id,
  start_date,
  end_date,
  encounter_month,
  previous_end_date,
  previous_encounter_id,
  DATEDIFF(start_date, previous_end_date) AS days_since_previous,
  CASE 
    WHEN previous_encounter_id IS NOT NULL 
         AND start_date >= previous_end_date 
    THEN 1 
    ELSE 0 
  END AS is_eligible,
  
  CASE 
    WHEN previous_encounter_id IS NOT NULL 
         AND start_date >= previous_end_date 
         AND DATEDIFF(start_date, previous_end_date) <= 30 
    THEN 1 
    ELSE 0 
  END AS is_readmission
FROM encounters_with_previous;

In [0]:
%sql

CREATE OR REPLACE TABLE medical_pipeline.gold.kpi_30day_readmission_rate AS
SELECT 
  DATE_FORMAT(encounter_month, 'yyyy-MM') AS month,
  SUM(is_eligible) AS eligible_encounters,
  SUM(is_readmission) AS readmissions,
  CASE 
    WHEN SUM(is_eligible) > 0 
    THEN ROUND(SUM(is_readmission) * 100.0 / SUM(is_eligible), 2)
    ELSE 0 
  END AS readmission_rate_pct
FROM classified_encounters
GROUP BY encounter_month
ORDER BY month;


In [0]:
%sql
SELECT * FROM medical_pipeline.gold.kpi_30day_readmission_rate;

In [0]:
%sql

SELECT 
  patient_id,
  encounter_id,
  start_date,
  end_date,
  previous_end_date,
  days_since_previous,
  is_eligible,
  is_readmission,
  CASE 
    WHEN is_readmission = 1 THEN 'Readmission'
    WHEN is_eligible = 1 THEN 'Eligible '
    ELSE 'Not Eligible'
  END AS status
FROM classified_encounters
WHERE patient_id IN (
  -
  SELECT patient_id 
  FROM classified_encounters 
  WHERE is_readmission = 1 
  LIMIT 5
)
ORDER BY patient_id, start_date
LIMIT 50;